# 09b. 긍정 리뷰 LLM 분류 + Aspect 시각화

## 목적
1. 긍정 리뷰를 층화 샘플링하여 Sonnet으로 aspect 분류
2. 긍정 유저가 칭찬하는 요소 vs 아쉬워하는 요소 파악
3. 부정 유저가 인정하는 요소 파악 (08편 결과 활용)
4. **aspect별 긍정/부정 교차 시각화** (산키 다이어그램 or 히트맵)

## 사전 조건
- `08_llm_classification.ipynb` 완료 (neg_reviews_classified 존재)
- `ANTHROPIC_API_KEY` 환경변수 설정

## 비용 추정
- 영어 3,000건 + 한국어 1,000건 = 4,000건
- Sonnet ($3/$15 per MTok): ~$5.20

## 긍정 리뷰 분류 체계
| aspect | 설명 |
|--------|------|
| atmosphere | 분위기, 아트, 음악, 비주얼 |
| gameplay_fun | 낚시, 다이빙, 전투가 재미있다 |
| story_charm | 스토리, 캐릭터, 유머 |
| variety | 다양한 콘텐츠 조합 (낚시+요리+경영) |
| value | 가성비, 플레이타임 대비 만족 |
| polish | 완성도, 버그 없음, 쾌적함 |
| other_positive | 위에 해당 안 되는 칭찬 |

추가 메타데이터 (부정 분류와 동일 구조):
- sub_aspect: 2차 칭찬 요소
- has_criticism: 긍정 리뷰 안에 비판 요소가 있는지 (yes/no)
- criticism_topic: 있다면 어떤 불만? (08편 카테고리 재사용)

In [1]:
import sys, os, sqlite3, json, time, re
import pandas as pd
import numpy as np
import platform, warnings
from tqdm import tqdm

import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
if platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
elif platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
tqdm.pandas()

sys.path.append('..')
from config import DB_PATH

db_path = os.path.join('..', DB_PATH)
conn = sqlite3.connect(db_path)

print(f'DB 연결: {db_path}')

DB 연결: ../data/dave_diver.db


In [2]:
def load_pos_reviews(conn, cleaned_table: str):
    """cleaned_reviews + reviews 원본 메타데이터 JOIN"""
    return pd.read_sql(f"""
        SELECT
            c.review_id,
            c.voted_up,
            c.review_month,
            c.play_segment,
            c.playtime_at_review,
            c.playtime_hours,
            c.received_for_free,
            c.written_during_early_access,
            c.review_text,
            c.cleaned_text,
            r.language,
            r.votes_up,
            r.votes_funny,
            r.weighted_vote_score,
            r.steam_purchase,
            r.developer_response,
            r.timestamp_dev_responded,
            r.num_games_owned,
            r.num_reviews
        FROM {cleaned_table} c
        LEFT JOIN reviews r ON c.review_id = r.review_id
        WHERE c.voted_up = 1
          AND c.cleaned_text IS NOT NULL
          AND c.cleaned_text != ''
    """, conn)


df_pos_en = load_pos_reviews(conn, 'cleaned_reviews_en')
df_pos_ko = load_pos_reviews(conn, 'cleaned_reviews_ko')

for df in [df_pos_en, df_pos_ko]:
    df['has_dev_response'] = df['developer_response'].notna().astype(int)
    df['text_len'] = df['review_text'].str.len()

print(f'영어 부정 리뷰:   {len(df_pos_en):,}건')
print(f'한국어 부정 리뷰: {len(df_pos_ko):,}건')
print(f'합계:             {len(df_pos_en) + len(df_pos_ko):,}건')

영어 부정 리뷰:   43,190건
한국어 부정 리뷰: 9,776건
합계:             52,966건


In [3]:
for name, df in [('영어', df_pos_en), ('한국어', df_pos_ko)]:
    total = len(df)
    over_500  = (df['text_len'] > 500).sum()
    over_1000 = (df['text_len'] > 1000).sum()
    over_2000 = (df['text_len'] > 2000).sum()
    
    print(f"\n[{name} 부정 리뷰] 총 {total:,}건")
    print(f"  500자 초과:  {over_500:,}건 ({over_500/total*100:.1f}%)")
    print(f"  1000자 초과: {over_1000:,}건 ({over_1000/total*100:.1f}%)")
    print(f"  2000자 초과: {over_2000:,}건 ({over_2000/total*100:.1f}%)")
    print(f"  평균 길이:   {df['text_len'].mean():.0f}자")
    print(f"  중앙값:      {df['text_len'].median():.0f}자")
    print(f"  최대:        {df['text_len'].max():,}자")


[영어 부정 리뷰] 총 43,190건
  500자 초과:  3,852건 (8.9%)
  1000자 초과: 1,399건 (3.2%)
  2000자 초과: 408건 (0.9%)
  평균 길이:   203자
  중앙값:      81자
  최대:        7,998자

[한국어 부정 리뷰] 총 9,776건
  500자 초과:  231건 (2.4%)
  1000자 초과: 79건 (0.8%)
  2000자 초과: 26건 (0.3%)
  평균 길이:   80자
  중앙값:      28자
  최대:        7,999자


---
## 2. 층화 샘플링

전체 긍정 리뷰에서 플레이타임 구간별로 고르게 추출.

| 구간 | 영어 | 한국어 |
|------|------|--------|
| casual(<2h) | 500 | 150 |
| regular(2-10h) | 1,000 | 350 |
| engaged(10-30h) | 1,000 | 350 |
| engaged(30-50h) | 250 | 75 |
| hardcore(50h+) | 250 | 75 |
| **합계** | **3,000** | **1,000** |

In [4]:
SAMPLE_SIZES_EN = {
    'casual(<2h)': 500,
    'regular(2-10h)': 1000,
    'engaged(10-30h)': 1000,
    'engaged(30-50h)': 250,
    'hardcore(50h+)': 250,
}
SAMPLE_SIZES_KO = {
    'casual(<2h)': 150,
    'regular(2-10h)': 350,
    'engaged(10-30h)': 350,
    'engaged(30-50h)': 75,
    'hardcore(50h+)': 75,
}


def stratified_sample(conn, cleaned_table, sample_sizes, seed=42):
    """플레이타임 구간별 층화 샘플링 (긍정 리뷰만)"""
    df = pd.read_sql(f"""
        SELECT c.review_id, c.voted_up, c.review_month, c.play_segment,
               c.playtime_hours, c.review_text, c.cleaned_text,
               r.votes_up, r.weighted_vote_score, r.steam_purchase,
               r.received_for_free, r.language
        FROM {cleaned_table} c
        LEFT JOIN reviews r ON c.review_id = r.review_id
        WHERE c.voted_up = 1
          AND c.cleaned_text IS NOT NULL AND c.cleaned_text != ''
    """, conn)

    samples = []
    for segment, n in sample_sizes.items():
        subset = df[df['play_segment'] == segment]
        actual_n = min(n, len(subset))
        sampled = subset.sample(n=actual_n, random_state=seed)
        samples.append(sampled)
        print(f'  {segment:20s}: {actual_n:>5} / {len(subset):>6} 추출')

    return pd.concat(samples, ignore_index=True)


print('=== 영어 긍정 리뷰 샘플링 ===')
pos_en = stratified_sample(conn, 'cleaned_reviews_en', SAMPLE_SIZES_EN)
print(f'영어 합계: {len(pos_en):,}건')

print(f'\n=== 한국어 긍정 리뷰 샘플링 ===')
pos_ko = stratified_sample(conn, 'cleaned_reviews_ko', SAMPLE_SIZES_KO)
print(f'한국어 합계: {len(pos_ko):,}건')

=== 영어 긍정 리뷰 샘플링 ===
  casual(<2h)         :   500 /   1429 추출
  regular(2-10h)      :  1000 /  11576 추출
  engaged(10-30h)     :  1000 /  14767 추출
  engaged(30-50h)     :   250 /   8636 추출
  hardcore(50h+)      :   250 /   6782 추출
영어 합계: 3,000건

=== 한국어 긍정 리뷰 샘플링 ===
  casual(<2h)         :   150 /    310 추출
  regular(2-10h)      :   350 /   2612 추출
  engaged(10-30h)     :   350 /   3851 추출
  engaged(30-50h)     :    75 /   1731 추출
  hardcore(50h+)      :    75 /   1272 추출
한국어 합계: 1,000건


---
## 3. 긍정 리뷰 분류 프롬프트 & 함수

In [5]:

#리뷰의 긍정요소 7가지 주요 카테고리 목록을 정의
#분위기, 게임플레이 재미, 스토리의 매력, 다양성, 가치, 완성도, 기타 긍정
POS_ASPECTS = ['atmosphere', 'gameplay_fun', 'story_charm', 'variety', 'value', 'polish', 'other_positive']

#부정 카테고리(이전파일에 참고)
NEG_CATEGORIES = ['gameplay', 'story_content', 'repetition', 'technical', 'company', 'forced_neg', 'other']
VALID_POS = set(POS_ASPECTS)

POS_DB_TABLE = 'pos_reviews_classified'


def build_positive_prompt(reviews_batch: list[dict], language: str) -> str:
    lang_note = ('리뷰는 한국어입니다. 한국어 맥락과 은어(갓겜, 꿀잼, 인생겜 등)를 고려하세요.'
                 if language == 'korean'
                 else 'Reviews are in English.')

    reviews_text = ''
    for item in reviews_batch:
        text = item['text'][:1000].replace('"', "'")
        reviews_text += f'[{item["idx"]}] {text}\n\n'

    prompt = f"""You are an expert game review analyst classifying POSITIVE Steam reviews of "Dave the Diver".
{lang_note}

## TASK
For each review, determine:
1. **aspect**: The PRIMARY thing being praised
2. **sub_aspect**: Secondary praise if present, or "none"
3. **has_criticism**: Does this positive review also contain criticism? (yes/no)
4. **criticism_topic**: If yes, what's the criticism about? (one of: gameplay, story_content, repetition, technical, company, other, none)
5. **confidence** and **reason**

## Positive Aspects

atmosphere: Art style, music, sound design, visual charm, underwater aesthetics, cozy vibes.
gameplay_fun: Fishing is fun, diving is exciting, combat is satisfying, minigames are enjoyable.
story_charm: Story is engaging, characters are lovable, humor is great, Bancho is funny.
variety: Mix of genres (fishing + cooking + restaurant + story), never boring, so much to do.
value: Worth the price, got many hours, great value for money, cheap for what you get.
polish: Bug-free, smooth experience, well-optimized, quality of life features.
other_positive: General praise that doesn't fit above.

## Key patterns
- "This game has everything" -> variety
- "Beautiful pixel art" -> atmosphere
- "Bancho made me laugh" -> story_charm
- "Best $20 I ever spent" -> value
- "Great game BUT the boss fights..." -> aspect=best fit, has_criticism=yes, criticism_topic=gameplay
- "Love it but gets repetitive" -> aspect=best fit, has_criticism=yes, criticism_topic=repetition

## Reviews

{reviews_text}

## Output
JSON array only. Each element:
- "idx": review number
- "aspect": one of [atmosphere, gameplay_fun, story_charm, variety, value, polish, other_positive]
- "sub_aspect": same options OR "none"
- "has_criticism": "yes" or "no"
- "criticism_topic": one of [gameplay, story_content, repetition, technical, company, other, none]
- "confidence": high / medium / low
- "reason": 1-sentence explanation

Return {len(reviews_batch)} items. JSON array only, no markdown fences."""

    return prompt


def parse_positive_response(response_text: str, expected_count: int) -> list[dict]:
    default = {
        'aspect': 'other_positive', 'sub_aspect': 'none',
        'has_criticism': 'no', 'criticism_topic': 'none',
        'confidence': 'low', 'reason': 'parse_failed',
    }

    def validate(item):
        asp = item.get('aspect', 'other_positive')
        sub = item.get('sub_aspect', 'none')
        return {
            'aspect': asp if asp in VALID_POS else 'other_positive',
            'sub_aspect': sub if (sub in VALID_POS or sub == 'none') else 'none',
            'has_criticism': item.get('has_criticism', 'no'),
            'criticism_topic': item.get('criticism_topic', 'none'),
            'confidence': item.get('confidence', 'unknown'),
            'reason': str(item.get('reason', ''))[:200],
        }

    cleaned = response_text.strip()
    cleaned = re.sub(r'^```(?:json)?\s*', '', cleaned)
    cleaned = re.sub(r'\s*```$', '', cleaned)

    try:
        results = json.loads(cleaned)
        if isinstance(results, list):
            validated = [validate(item) for item in results]
            while len(validated) < expected_count:
                validated.append({**default, 'reason': 'missing'})
            return validated[:expected_count]
    except json.JSONDecodeError:
        pass

    json_match = re.search(r'\[.*\]', cleaned, re.DOTALL)
    if json_match:
        try:
            results = json.loads(json_match.group())
            if isinstance(results, list):
                validated = [validate(item) for item in results]
                while len(validated) < expected_count:
                    validated.append({**default, 'reason': 'partial'})
                return validated[:expected_count]
        except json.JSONDecodeError:
            pass

    return [dict(default) for _ in range(expected_count)]


print('긍정 분류 프롬프트 & 파서 정의 완료')

긍정 분류 프롬프트 & 파서 정의 완료


In [6]:
def classify_positive_batch(reviews_batch: list[dict], language: str,
                             model: str = 'claude-sonnet-4-5-20250929') -> list[dict]:
    """Anthropic Sonnet으로 긍정 리뷰 배치 분류"""
    import anthropic
    client = anthropic.Anthropic()

    prompt = build_positive_prompt(reviews_batch, language)
    response = client.messages.create(
        model=model,
        max_tokens=4096,
        messages=[{'role': 'user', 'content': prompt}]
    )

    return parse_positive_response(response.content[0].text, len(reviews_batch))


print('Anthropic 긍정 분류 함수 정의 완료')

Anthropic 긍정 분류 함수 정의 완료


---
## 4. DB 저장 & 분류 실행

In [7]:
def init_pos_table(conn):
    conn.execute(f"""
        CREATE TABLE IF NOT EXISTS {POS_DB_TABLE} (
            review_id           TEXT PRIMARY KEY,
            language_group      TEXT,
            play_segment        TEXT,
            playtime_hours      REAL,
            review_month        TEXT,
            votes_up            INTEGER,
            weighted_vote_score REAL,
            steam_purchase      INTEGER,
            received_for_free   INTEGER,
            review_text         TEXT,
            cleaned_text        TEXT,
            -- 분류 결과
            aspect              TEXT,
            sub_aspect          TEXT,
            has_criticism        TEXT,
            criticism_topic     TEXT,
            confidence          TEXT,
            reason              TEXT
        )
    """)
    conn.commit()


def get_pos_already(conn) -> set:
    try:
        result = pd.read_sql(f"SELECT review_id FROM {POS_DB_TABLE}", conn)
        return set(result['review_id'].tolist())
    except Exception:
        return set()


def save_pos_batch(conn, df_batch, results, lang_group):
    rows = []
    for i, (_, row) in enumerate(df_batch.iterrows()):
        r = results[i] if i < len(results) else {
            'aspect': 'other_positive', 'sub_aspect': 'none',
            'has_criticism': 'no', 'criticism_topic': 'none',
            'confidence': 'low', 'reason': 'error',
        }
        rows.append({
            'review_id': row['review_id'],
            'language_group': lang_group,
            'play_segment': row.get('play_segment'),
            'playtime_hours': row.get('playtime_hours'),
            'review_month': row.get('review_month'),
            'votes_up': row.get('votes_up'),
            'weighted_vote_score': row.get('weighted_vote_score'),
            'steam_purchase': row.get('steam_purchase'),
            'received_for_free': row.get('received_for_free'),
            'review_text': row['review_text'],
            'cleaned_text': row['cleaned_text'],
            'aspect': r['aspect'],
            'sub_aspect': r.get('sub_aspect', 'none'),
            'has_criticism': r.get('has_criticism', 'no'),
            'criticism_topic': r.get('criticism_topic', 'none'),
            'confidence': r['confidence'],
            'reason': r['reason'],
        })
    pd.DataFrame(rows).to_sql(POS_DB_TABLE, conn, if_exists='append', index=False)
    conn.commit()


def run_positive_classification(conn, df_pos, lang_group, batch_size=25):
    already = get_pos_already(conn)
    remaining = df_pos[~df_pos['review_id'].isin(already)].copy()

    if len(remaining) == 0:
        print(f'  [{lang_group}] 이미 전부 완료 ({len(df_pos):,}건). 건너뜀.')
        return

    print(f'  [{lang_group}] 분류 대상: {len(remaining):,}건 (이미 완료: {len(already & set(df_pos["review_id"])):,}건)')

    total_batches = (len(remaining) + batch_size - 1) // batch_size

    for start in tqdm(range(0, len(remaining), batch_size),
                      total=total_batches, desc=f'{lang_group} (sonnet)'):
        batch_df = remaining.iloc[start:start+batch_size]
        batch_input = [{'idx': j+1, 'text': row['review_text']}
                       for j, (_, row) in enumerate(batch_df.iterrows())]

        try:
            results = classify_positive_batch(batch_input, lang_group)
            save_pos_batch(conn, batch_df, results, lang_group)
        except Exception as e:
            print(f'\n  [에러] batch {start//batch_size}: {e}')

        time.sleep(1.0)


print('긍정 분류 엔진 정의 완료')

긍정 분류 엔진 정의 완료


In [8]:
init_pos_table(conn)

print('=== 긍정 리뷰 분류 시작 (Sonnet) ===')
run_positive_classification(conn, pos_en, 'english')
run_positive_classification(conn, pos_ko, 'korean')

check = pd.read_sql(f"""
    SELECT language_group, COUNT(*) as n
    FROM {POS_DB_TABLE}
    GROUP BY language_group
""", conn)
print(f'\n{check.to_string(index=False)}')

=== 긍정 리뷰 분류 시작 (Sonnet) ===
  [english] 이미 전부 완료 (3,000건). 건너뜀.
  [korean] 이미 전부 완료 (1,000건). 건너뜀.

language_group    n
       english 3000
        korean 1000


---
## 5. 긍정 분류 결과 분석

In [9]:
pos_cls = pd.read_sql(f"SELECT * FROM {POS_DB_TABLE}", conn)
pos_en_cls = pos_cls[pos_cls['language_group'] == 'english'].copy()
pos_ko_cls = pos_cls[pos_cls['language_group'] == 'korean'].copy()

print(f'긍정 영어: {len(pos_en_cls):,}건, 한국어: {len(pos_ko_cls):,}건')

# aspect 분포
print(f'\n=== aspect 분포 (영어) ===')
for asp, cnt in pos_en_cls['aspect'].value_counts().items():
    pct = cnt / len(pos_en_cls) * 100
    bar = '█' * int(pct / 2)
    print(f'  {asp:20s} {cnt:>5}건 ({pct:>5.1f}%) {bar}')

# 긍정 리뷰 안의 비판
has_crit = pos_en_cls[pos_en_cls['has_criticism'] == 'yes']
print(f'\n=== 긍정인데 비판 포함 (영어) ===')
print(f'{len(has_crit):,}건 / {len(pos_en_cls):,}건 ({len(has_crit)/len(pos_en_cls)*100:.1f}%)')
if len(has_crit) > 0:
    print(f'\n비판 주제:')
    print(has_crit['criticism_topic'].value_counts().to_string())
    
    # aspect 분포
print(f'\n=== aspect 분포 (한국어) ===')
for asp, cnt in pos_ko_cls['aspect'].value_counts().items():
    pct = cnt / len(pos_ko_cls) * 100
    bar = '█' * int(pct / 2)
    print(f'  {asp:20s} {cnt:>5}건 ({pct:>5.1f}%) {bar}')

# 긍정 리뷰 안의 비판
has_crit = pos_ko_cls[pos_ko_cls['has_criticism'] == 'yes']
print(f'\n=== 긍정인데 비판 포함 (한국어) ===')
print(f'{len(has_crit):,}건 / {len(pos_ko_cls):,}건 ({len(has_crit)/len(pos_ko_cls)*100:.1f}%)')
if len(has_crit) > 0:
    print(f'\n비판 주제:')
    print(has_crit['criticism_topic'].value_counts().to_string())

긍정 영어: 3,000건, 한국어: 1,000건

=== aspect 분포 (영어) ===
  other_positive         914건 ( 30.5%) ███████████████
  gameplay_fun           665건 ( 22.2%) ███████████
  variety                620건 ( 20.7%) ██████████
  atmosphere             446건 ( 14.9%) ███████
  story_charm            219건 (  7.3%) ███
  value                   73건 (  2.4%) █
  polish                  63건 (  2.1%) █

=== 긍정인데 비판 포함 (영어) ===
299건 / 3,000건 (10.0%)

비판 주제:
criticism_topic
gameplay         89
other            65
repetition       59
technical        43
story_content    35
company           8

=== aspect 분포 (한국어) ===
  other_positive         355건 ( 35.5%) █████████████████
  gameplay_fun           298건 ( 29.8%) ██████████████
  variety                144건 ( 14.4%) ███████
  atmosphere             101건 ( 10.1%) █████
  story_charm             39건 (  3.9%) █
  value                   38건 (  3.8%) █
  polish                  25건 (  2.5%) █

=== 긍정인데 비판 포함 (한국어) ===
228건 / 1,000건 (22.8%)

비판 주제:
criticism_topic
gamepla

---
## 6. Aspect별 긍정/부정 교차 시각화

핵심 시각화: "초밥집은 긍정, 인어마을 스토리는 부정" 같은 패턴을
긍정 유저와 부정 유저 양쪽에서 비교.

### 매핑
긍정 aspect와 부정 category를 공통 게임 요소로 매핑:

| 게임 요소 | 긍정 aspect | 부정 category |
|----------|------------|---------------|
| 게임플레이 | gameplay_fun | gameplay |
| 스토리 | story_charm | story_content |
| 분위기/아트 | atmosphere | (해당 없음) |
| 콘텐츠 다양성 | variety | repetition (반대) |
| 가성비 | value | company |
| 완성도 | polish | technical |

In [10]:
# ── 부정 분류 결과 로드 ──
neg_cls = pd.read_sql("""
    SELECT * FROM neg_reviews_classified
    WHERE classify_method = 'anthropic' AND language_group = 'english'
""", conn)

# ── 공통 요소로 매핑 ──
ELEMENT_MAP_POS = {
    'gameplay_fun': '게임플레이',
    'story_charm': '스토리',
    'atmosphere': '분위기/아트',
    'variety': '콘텐츠 다양성',
    'value': '가성비',
    'polish': '완성도',
}
ELEMENT_MAP_NEG = {
    'gameplay': '게임플레이',
    'story_content': '스토리',
    'repetition': '콘텐츠 다양성',  # 반대 관계
    'company': '가성비',
    'technical': '완성도',
}

# 긍정 유저의 element별 비중
pos_mapped = pos_en_cls['aspect'].map(ELEMENT_MAP_POS).dropna().value_counts(normalize=True) * 100

# 부정 유저의 element별 비중
neg_mapped = neg_cls['category'].map(ELEMENT_MAP_NEG).dropna().value_counts(normalize=True) * 100

# 합치기
elements = sorted(set(pos_mapped.index) | set(neg_mapped.index))
cross_df = pd.DataFrame({
    '긍정 유저가 칭찬 (%)': pos_mapped.reindex(elements, fill_value=0),
    '부정 유저가 불만 (%)': neg_mapped.reindex(elements, fill_value=0),
}).round(1)

print('=== 게임 요소별 긍정/부정 교차 ===')
print(cross_df.to_string())

=== 게임 요소별 긍정/부정 교차 ===
         긍정 유저가 칭찬 (%)  부정 유저가 불만 (%)
가성비                3.5           11.2
게임플레이             31.9           35.6
분위기/아트            21.4            0.0
스토리               10.5           11.6
완성도                3.0           11.9
콘텐츠 다양성           29.7           29.8


In [11]:
# ── 나비 차트 (Butterfly Chart): 왼쪽 긍정, 오른쪽 부정 ──
fig = go.Figure()

fig.add_trace(go.Bar(
    y=elements, x=[-v for v in cross_df['긍정 유저가 칭찬 (%)']],
    orientation='h', name='긍정 유저가 칭찬',
    marker_color='#4ECDC4',
    text=[f'{v:.0f}%' for v in cross_df['긍정 유저가 칭찬 (%)']],
    textposition='outside'))

fig.add_trace(go.Bar(
    y=elements, x=cross_df['부정 유저가 불만 (%)'],
    orientation='h', name='부정 유저가 불만',
    marker_color='#FF6B6B',
    text=[f'{v:.0f}%' for v in cross_df['부정 유저가 불만 (%)']],
    textposition='outside'))

fig.update_layout(
    title='게임 요소별 긍정/부정 교차 분석 (나비 차트)',
    xaxis_title='비중 (%)',
    barmode='overlay',
    height=400, width=900, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()

In [12]:
# ── 긍정 유저의 비판 요소 ("좋지만 이건 아쉬워") ──
if len(has_crit) > 0:
    crit_by_segment = pd.crosstab(
        has_crit['play_segment'], has_crit['criticism_topic'],
        normalize='index'
    ) * 100

    print('=== 긍정 유저의 비판 토픽 (구간별) ===')
    print(crit_by_segment.round(1).to_string())

    # 샘플
    print(f'\n--- "좋지만 이건 아쉬워" 샘플 5건 ---')
    for _, row in has_crit.head(5).iterrows():
        print(f'[{row["play_segment"]}] [칭찬={row["aspect"]}] [비판={row["criticism_topic"]}]')
        print(f'  {row["review_text"][:800]}')
        print()

=== 긍정 유저의 비판 토픽 (구간별) ===
criticism_topic  company  gameplay  other  repetition  story_content  technical
play_segment                                                                   
casual(<2h)         21.6      27.0   18.9         2.7            2.7       27.0
engaged(10-30h)      9.8      23.9   17.4        28.3            9.8       10.9
engaged(30-50h)      8.7      21.7    4.3        34.8           26.1        4.3
hardcore(50h+)       0.0      15.8   31.6        42.1           10.5        0.0
regular(2-10h)       1.8      31.6   22.8        15.8            7.0       21.1

--- "좋지만 이건 아쉬워" 샘플 5건 ---
[casual(<2h)] [칭찬=other_positive] [비판=gameplay]
  크~ 이게 게임이지~ bb 
이런류 게임은 휴대용기기로 즐기는 맛이 있어서 스위치버전 나올때까지 기다렸는데,
막상 스위치 출시되니 스위치는 역시 pc에 비해 로딩이.....ㅠㅠ
이왕 구매 늦은거 할인때까지 기다리다 구매~ㅎ
앞으로 이런 게임다운 게임 좀 많이 만들어주길~~~ ㅋㅋ

근데 뚱보새리 단련하거나 스탯 찍는건 없나? 초밥 서빙 ㅈㄴ느리네....
개답답...

[casual(<2h)] [칭찬=atmosphere] [비판=story_content]
  아직 얼리엑세스여서 그런지 모르겠는데 재미는 있지만 할수 있는게 적은 느낌이었음,
바다를 돌아다니머 편안하게 게임을 하고 싶다 하면 이게임

---
## 7. 플레이타임 구간별 칭찬 요소 변화

In [13]:
segment_order = ['casual(<2h)', 'regular(2-10h)', 'engaged(10-30h)', 'engaged(30-50h)', 'hardcore(50h+)']

seg_aspect = pd.crosstab(pos_en_cls['play_segment'], pos_en_cls['aspect'], normalize='index') * 100
seg_aspect = seg_aspect.reindex(index=segment_order, columns=POS_ASPECTS).fillna(0)

fig = go.Figure()
pos_colors = {
    'atmosphere': '#DDA0DD', 'gameplay_fun': '#FF6B6B', 'story_charm': '#4ECDC4',
    'variety': '#45B7D1', 'value': '#FFEAA7', 'polish': '#96CEB4', 'other_positive': '#C0C0C0'
}
for asp in POS_ASPECTS:
    fig.add_trace(go.Bar(
        name=asp, x=segment_order,
        y=[seg_aspect.loc[s, asp] if s in seg_aspect.index else 0 for s in segment_order],
        marker_color=pos_colors.get(asp, '#999')))

fig.update_layout(
    barmode='stack',
    title='플레이타임 구간별 칭찬 요소 구성',
    xaxis_title='플레이타임', yaxis_title='비중 (%)',
    height=500, width=900, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02))
fig.show()

---
## 결론

*(실행 후 작성)*

### 핵심 발견
1. 긍정 유저가 가장 많이 칭찬하는 요소: ___
2. 긍정 유저도 아쉬워하는 요소: ___
3. 라이트 유저 vs 하드코어 유저의 칭찬 포인트 차이: ___
4. 긍정/부정 교차 핵심: ___

### 가설 2 심화 결론
- 라이트 유저(2-10h)가 높은 긍정률을 보이는 이유: ___

In [14]:
conn.close()
print('DB 연결 종료')

DB 연결 종료
